In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from ddcc.orientationscore import cakewavelet_stack, wavelet_transform
from ddcc.vesselness import multi_scale_vesselness, field_of_view_mask
from ddcc.algorithms import (
    connected_components,
    persistence,
    compare_components,
    crossing_mask,
    project_components,
)
from pathlib import Path
from PIL import Image

plt.rcParams.update({
    # "text.usetex": True,
    # "font.family": "serif",
    # "font.serif": ["Computer Modern"],
    # "font.size": 10.0,
    # "text.latex.preamble": r"\usepackage{lmodern} \usepackage{amssymb} \usepackage{amsmath}",
    "image.cmap": "gray",
    "image.interpolation": "nearest",
    "figure.constrained_layout.use": True,
    "xtick.top": False,
    "xtick.bottom": False,
    "ytick.left": False,
    "ytick.right": False,
    "xtick.labeltop": False,
    "xtick.labelbottom": False,
    "ytick.labelleft": False,
    "ytick.labelright": False,
})
latex_line_width = 4.804

device = "cuda"

In [ ]:
# colours
red = "#B6321C" # BrickRed
blue = "#0071BC" # RoyalBlue
green = "#009B55" # ForestGreen
purple = "#613F99" # RoyalPurple

def plot_image(ax, image, **kwargs):
    image = image.cpu()
    return ax.imshow(image, **kwargs)

def plot_components(fig, ax, components, random_colors=False, **kwargs):
    N_components = components.max().item()
    colors = plt.cm.rainbow(np.linspace(0, 1, N_components))
    colors = np.concat([[[0, 0, 0, 1]], colors], axis=0)
    if random_colors:
        np.random.shuffle(colors)
    cmap = mcolors.ListedColormap(colors)
    im = ax.imshow(components.cpu(), cmap=cmap)
    fig.colorbar(im, ticks=np.arange(1, N_components+1), ax=ax)
    return im

output_directory = Path("output")
data_directory = Path("data")
config_path = Path("config.json")

In [ ]:
def load_image(identifier, data_directory=data_directory, device=device, downsample=1):
    """Load retinal image `##_???.jpg` as float grayscale tensor of shape [H, W] in [0, 1].
    Downsample by averaging over blocks of `downsample` x `downsample` pixels."""
    image = np.array(Image.open(data_directory / f"{identifier}.jpg").convert("L"))
    image = torch.tensor(image, device=device, dtype=torch.float32) / 255.
    return F.avg_pool2d(image[None], downsample)[0]

def load_segmentation(identifier, data_directory=data_directory, device=device, threshold=0.5, downsample=1):
    """Load segmentation `##_???_segmented.jpg` as binary float tensor of shape [H, W].
    Downsample by max pooling, so that thin vessels are preserved."""
    segmentation = np.array(Image.open(data_directory / f"{identifier}_segmented.jpg").convert("L"))
    segmentation = (torch.tensor(segmentation, device=device, dtype=torch.float32) / 255. > threshold) * 1.
    return F.max_pool2d(segmentation[None], downsample)[0]

def load_components(identifier, data_directory=data_directory, device=device, downsample=1):
    """Load components `##_???_components.txt` as integer label tensor of shape [H, W], with 0 background.
    Downsample by max pooling, so that the foreground matches the downsampled segmentation."""
    components = np.loadtxt(data_directory / f"{identifier}_components.txt", delimiter=",", dtype=np.int64)
    components = torch.tensor(components, device=device, dtype=torch.float32)
    return F.max_pool2d(components[None], downsample)[0].long()

def load_data(identifier, data_directory=data_directory, device=device, downsample=1):
    """Load image, segmentation, and components belonging to `identifier`, e.g. "01_OSC"."""
    return (
        load_image(identifier, data_directory, device, downsample=downsample),
        load_segmentation(identifier, data_directory, device, downsample=downsample),
        load_components(identifier, data_directory, device, downsample=downsample),
    )

In [ ]:
downsample = 1
image, segmentation, components_gt = load_data("01_OSC", downsample=downsample)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
plot_image(ax[0], image)
plot_image(ax[1], segmentation)
plot_components(fig, ax[2], components_gt);

In [ ]:
H, W = image.shape
Or = 32

In [ ]:
cws = cakewavelet_stack(min(H, W), Or).to("cuda")
U = wavelet_transform(image, cws)

In [ ]:
K = 0
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
cbar = plot_image(ax[0], cws[K])
fig.colorbar(cbar, ax=ax[0])
cbar = plot_image(ax[1], U[K])
fig.colorbar(cbar, ax=ax[1]);

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
cbar = plot_image(ax[0], image)
fig.colorbar(cbar, ax=ax[0])
cbar = plot_image(ax[1], torch.clip(U.sum(0), 0., 1.))
fig.colorbar(cbar, ax=ax[1]);

In [ ]:
mask = field_of_view_mask(image, erosion=10)

In [ ]:
σs_s_list = (0.5, 1., 2., 4.)
σ_0 = 0.5
σ_s_ext = 1.
σ_o_ext = 0.5
V = multi_scale_vesselness(U, σs_s_list, σ_0, σ_s_ext, σ_o_ext, mask)
λ = 100
p = 2
C = 1 / (1 + λ * V**p)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
cbar = plot_image(ax[0], segmentation)
fig.colorbar(cbar, ax=ax[0])
cbar = plot_image(ax[1], C.min(0)[0])
fig.colorbar(cbar, ax=ax[1]);

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 10))
cbar = plot_image(ax[0, 0], C[0])
fig.colorbar(cbar, ax=ax[0, 0])
cbar = plot_image(ax[0, 1], C[4])
fig.colorbar(cbar, ax=ax[0, 1])
cbar = plot_image(ax[1, 0], C[8])
fig.colorbar(cbar, ax=ax[1, 0])
cbar = plot_image(ax[1, 1], C[12])
fig.colorbar(cbar, ax=ax[1, 1]);

In [ ]:
on_data = torch.stack(Or*[segmentation])

In [ ]:
offset = 0
fig, ax = plt.subplots(2, 2, figsize=(12, 10))
cbar = plot_image(ax[0, 0], on_data[0+offset])
fig.colorbar(cbar, ax=ax[0, 0])
cbar = plot_image(ax[0, 1], on_data[4+offset])
fig.colorbar(cbar, ax=ax[0, 1])
cbar = plot_image(ax[1, 0], on_data[8+offset])
fig.colorbar(cbar, ax=ax[1, 0])
cbar = plot_image(ax[1, 1], on_data[12+offset])
fig.colorbar(cbar, ax=ax[1, 1]);

In [ ]:
plt.imshow(on_data.amax(0).cpu())

In [ ]:
components = connected_components(on_data, C, 0.01, (1., 0., 0.00025), threshold=0.8)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
plot_components(fig, ax, components.amax(0))

In [ ]:
distance_map, persistence_diagram = persistence(on_data, C, torch.linspace(0, 5, 10), (1., 0., 0.00025), n_max=6000)

In [ ]:
fig, ax = plt.subplots(1, 1)
cbar = plot_image(ax, distance_map.amin(0) < 10.)
fig.colorbar(cbar, ax=ax);

In [ ]:
output = compare_components(
    project_components(
        components, 
        V
    )[0],
    components_gt,
    crossing_mask(components_gt),
)

In [ ]:
output["accuracy"]